# Introduction to Arabic Speech Technologies
## Chapter 6 notebook: The self-supervised objectives, on toy data

Electronic supplementary material for *Introduction to Arabic Speech Technologies* by Hend S. Al-Khalifa.

Chapter 6 explains what wav2vec 2.0, HuBERT and predictive coding learn from unlabelled audio without deriving their loss functions, and leaves the formal objectives to the cited papers and to this notebook. The three losses are written out here in `numpy` on small synthetic tensors, so the shapes and the arithmetic can be inspected. The last section reproduces the shape of a layer-wise probing experiment of the kind the chapter describes.

**Contents**

1. Masking a sequence of frames
2. The contrastive objective of wav2vec 2.0 (InfoNCE with distractors)
3. Product quantization: the discrete targets
4. Masked prediction as classification (HuBERT and WavLM)
5. Layer-wise probing: what a frozen encoder makes decodable
6. A note on fine-tuning cost

**Running it.** The notebook needs only `numpy`, `scipy` and `matplotlib` (see `requirements.txt`). It runs top to bottom with no downloads and no audio files: where a recording is useful, the notebook synthesises one, and a cell is provided for reading your own WAV file instead. Optional cells that need extra packages or internet access are marked *Optional*.


## 1. Masking

Both families hide part of the input and ask the model to recover information about the hidden part. Spans of frames are chosen at random, as in the papers, rather than isolated frames.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

T, D, V = 60, 16, 32          # frames, feature size, codebook size
def span_mask(T, p_start=0.08, span=5, rng=rng):
    mask = np.zeros(T, dtype=bool)
    for t in range(T):
        if rng.random() < p_start:
            mask[t:t + span] = True
    return mask

mask = span_mask(T)
print(f"{mask.sum()} of {T} frames masked ({100*mask.mean():.0f}%), in spans of about 5 frames")
print("".join("M" if m else "." for m in mask))

## 2. The contrastive objective

For each masked position the model produces a context vector $c_t$ and must pick its own quantized target $q_t$ out of a set that also contains distractors sampled from other masked positions. With cosine similarity and temperature $\kappa$, the loss at position $t$ is

$$\mathcal{L}_t = -\log \frac{\exp(\mathrm{sim}(c_t, q_t)/\kappa)}{\sum_{\tilde q \in Q_t} \exp(\mathrm{sim}(c_t, \tilde q)/\kappa)}.$$

In [ ]:
def cosine(a, b):
    return (a @ b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12)

def info_nce(context, targets, masked_idx, n_distractors=10, kappa=0.1, rng=rng):
    losses, correct = [], 0
    for t in masked_idx:
        pool = [i for i in masked_idx if i != t]
        distractors = rng.choice(pool, size=min(n_distractors, len(pool)), replace=False)
        candidates = np.vstack([targets[t], targets[distractors]])
        sims = np.array([cosine(context[t], q) for q in candidates]) / kappa
        sims -= sims.max()
        p = np.exp(sims) / np.exp(sims).sum()
        losses.append(-np.log(p[0] + 1e-12))
        correct += int(np.argmax(sims) == 0)
    return float(np.mean(losses)), correct / len(masked_idx)

masked_idx = np.where(mask)[0]
targets = rng.standard_normal((T, D))
random_ctx = rng.standard_normal((T, D))
trained_ctx = targets + 0.35 * rng.standard_normal((T, D))    # a model that has learned something

for name, ctx in (("untrained context", random_ctx), ("trained context", trained_ctx)):
    loss, acc = info_nce(ctx, targets, masked_idx)
    print(f"{name:18}  InfoNCE loss {loss:5.3f}   picks its own target {100*acc:5.1f}% of the time")
print(f"\nchance level with 10 distractors: {100/11:.1f}%")

## 3. Product quantization

The targets are discrete. A product quantizer splits the feature vector into groups and picks one entry per group from a small codebook, so a handful of codebooks can describe many combinations. The cell shows the codebook usage, which is what the diversity term in the real loss is there to keep high.

In [ ]:
G, E = 2, V            # groups, entries per group
codebooks = rng.standard_normal((G, E, D // G))

def quantize(x):
    codes, pieces = [], np.split(x, G)
    for g, piece in enumerate(pieces):
        d = np.linalg.norm(codebooks[g] - piece, axis=1)
        codes.append(int(np.argmin(d)))
    return codes, np.concatenate([codebooks[g][c] for g, c in enumerate(codes)])

all_codes = np.array([quantize(f)[0] for f in targets])
usage = [len(set(all_codes[:, g])) for g in range(G)]
print(f"codebooks: {G} groups x {E} entries -> {E**G} possible combinations")
print("entries actually used per group:", usage)
print("first five frames, as code pairs:", [tuple(int(v) for v in c) for c in all_codes[:5]])

## 4. Masked prediction

HuBERT replaces the choice among distractors with plain classification: cluster features to obtain pseudo-labels, then predict the label of each masked frame with a cross-entropy loss. The first iteration clusters MFCC-like features; later iterations cluster the model's own representations.

In [ ]:
def kmeans(x, k, iters=25, rng=rng):
    centres = x[rng.choice(len(x), size=k, replace=False)]
    for _ in range(iters):
        assign = np.argmin(((x[:, None, :] - centres[None]) ** 2).sum(-1), axis=1)
        for j in range(k):
            if np.any(assign == j):
                centres[j] = x[assign == j].mean(axis=0)
    return centres, assign

K = 8
centres, labels = kmeans(targets, K)
print("pseudo-label counts:", np.bincount(labels, minlength=K))

def cross_entropy(logits, label):
    z = logits - logits.max()
    p = np.exp(z) / np.exp(z).sum()
    return -np.log(p[label] + 1e-12), int(np.argmax(logits))

W = rng.standard_normal((D, K)) * 0.3          # a linear prediction head
losses, hits = [], 0
for t in masked_idx:
    logits = trained_ctx[t] @ W
    loss, pred = cross_entropy(logits, labels[t])
    losses.append(loss); hits += int(pred == labels[t])
print(f"masked-prediction loss {np.mean(losses):.3f}, accuracy {100*hits/len(masked_idx):.1f}% "
      f"(chance {100/K:.1f}%) with an untrained head")

## 5. Layer-wise probing

A probe is a small classifier trained on frozen representations from one layer. The synthetic encoder below carries speaker identity in its early layers and phone identity in the middle ones, which is the pattern reported for wav2vec 2.0 and which Figure 6.4 illustrates. The probe is a logistic regression trained with plain gradient descent, so nothing beyond `numpy` is needed.

In [ ]:
n_items, n_layers = 400, 12
speaker = rng.integers(0, 4, n_items)
phone   = rng.integers(0, 6, n_items)
spk_dirs = rng.standard_normal((4, D)); ph_dirs = rng.standard_normal((6, D))

def layer_rep(l):
    spk_weight = np.exp(-((l - 2) ** 2) / 8)          # strongest early
    ph_weight  = np.exp(-((l - 7) ** 2) / 6)          # strongest in the middle
    return (spk_weight * spk_dirs[speaker] + ph_weight * ph_dirs[phone]
            + 0.8 * rng.standard_normal((n_items, D)))

def probe_accuracy(X, y, epochs=300, lr=0.5):
    classes = int(y.max()) + 1
    Xb = np.hstack([X, np.ones((len(X), 1))])
    split = int(0.7 * len(X))
    W = np.zeros((Xb.shape[1], classes))
    Y = np.eye(classes)[y]
    for _ in range(epochs):
        z = Xb[:split] @ W; z -= z.max(axis=1, keepdims=True)
        p = np.exp(z) / np.exp(z).sum(axis=1, keepdims=True)
        W -= lr * Xb[:split].T @ (p - Y[:split]) / split
    pred = np.argmax(Xb[split:] @ W, axis=1)
    return float((pred == y[split:]).mean())

print(f"{'layer':6} {'speaker probe':>14} {'phone probe':>13}")
for l in range(n_layers):
    X = layer_rep(l)
    print(f"{l:6d} {probe_accuracy(X, speaker):14.2f} {probe_accuracy(X, phone):13.2f}")
print("\nHigh accuracy means the property is decodable from that layer, not that the model uses it.")

## 6. Fine-tuning cost

Chapter 6 asks readers to compare full fine-tuning with adapters and LoRA. The arithmetic below is the parameter count only; memory and time also depend on the optimiser state and the batch.

In [ ]:
encoder_params = 300_000_000
d_model, r, n_layers_ft = 1024, 8, 24
lora = 2 * n_layers_ft * 2 * d_model * r          # two projections per layer, A and B matrices
adapters = n_layers_ft * (2 * d_model * 64 + 64 + d_model)
for name, count in (("full fine-tuning", encoder_params), ("adapters (bottleneck 64)", adapters), (f"LoRA (r={r})", lora)):
    print(f"{name:26} {count:>12,} trainable parameters  ({100*count/encoder_params:6.2f}% of the encoder)")